# LLM-as-a-Judge Evaluation Frameworks & Bias Mitigation

## 1. The Evaluation Bottleneck in AI Engineering
Traditional NLP metrics like BLEU and ROUGE evaluate n-gram lexical overlap. They correlate poorly with human judgment for complex agentic tasks:
- A syntactically perfect answer with a false fact receives high ROUGE but is disastrous in production.
- A completely correct answer phrased differently receives low ROUGE.

### The LLM-as-a-Judge Paradigm
Using powerful foundation models (e.g. GPT-4, Claude 3.5 Sonnet) as automated evaluators enables nuanced assessment of **Correctness, Coherence, Hallucination, and Safety**.

### The 3 Critical Biases of LLM Judges
1. **Position Bias**: Evaluators systematically prefer whichever candidate model response is placed first (Option A) in pairwise comparisons.
2. **Verbosity Bias**: Evaluators favor lengthy, verbose, formatting-heavy responses over concise, accurate ones.
3. **Self-Enhancement Bias**: Models often assign higher scores to responses generated by themselves.

### Mathematical Mitigation: Dual-Pass Order Swapping
To eliminate position bias, we execute a **two-pass evaluation** swapping candidate order:
- Pass 1: Compare $(A, B) \to \text{Win Score } S_1 \in \{0, 0.5, 1\}$
- Pass 2: Compare $(B, A) \to \text{Win Score } S_2 \in \{0, 0.5, 1\}$

$$\text{Unbiased Score}(A) = \frac{S_1 + (1 - S_2)}{2}$$

In [ ]:
import numpy as np

class SimulatedLLMJudge:
    """
    Demonstrates dual-pass pairwise evaluation with position bias mitigation.
    """
    def __init__(self, position_bias_strength=0.15):
        self.position_bias_strength = position_bias_strength

    def _judge_pair(self, resp_first, resp_second, true_first_score, true_second_score):
        """
        Simulates evaluation where candidate placed FIRST receives an artificial boost.
        """
        effective_first = true_first_score + self.position_bias_strength
        effective_second = true_second_score
        
        diff = effective_first - effective_second
        if diff > 0.1:
            return 1.0 # First wins
        elif diff < -0.1:
            return 0.0 # Second wins
        return 0.5 # Tie

    def evaluate_pairwise(self, model_a_text, model_b_text, quality_a, quality_b):
        # Pass 1: A is first, B is second
        s1 = self._judge_pair(model_a_text, model_b_text, quality_a, quality_b)
        
        # Pass 2: B is first, A is second (Order swapped!)
        s2 = self._judge_pair(model_b_text, model_a_text, quality_b, quality_a)
        
        # DebBiased score for Model A
        unbiased_a = (s1 + (1.0 - s2)) / 2.0
        return s1, s2, unbiased_a

# Simulation
judge = SimulatedLLMJudge(position_bias_strength=0.20)
# Model A and Model B are equally good (quality = 0.80)
s1, s2, final_score = judge.evaluate_pairwise("Model A answer", "Model B answer", 0.80, 0.80)

print("Pairwise Judge Evaluation with Synthetic Position Bias:")
print(f"  Pass 1 (Order: A then B) -> Model A Win Score : {s1} (Biased toward A)")
print(f"  Pass 2 (Order: B then A) -> Model B Win Score : {s2} (Biased toward B)")
print(f"  Calibrated Multi-Pass Balanced Score for A   : {final_score:.2f} (Perfect Fair Tie!)")

## 2. Inter-Annotator Agreement: Cohen's Kappa ($\kappa$)
To validate whether an automated LLM judge is aligned with human domain experts, we calculate Cohen's Kappa:

$$\kappa = \frac{p_o - p_e}{1 - p_e}$$

Where $p_o$ is observed agreement and $p_e$ is hypothetical chance agreement.

In [ ]:
def cohen_kappa(human_ratings, llm_ratings):
    """
    Computes Cohen's Kappa for binary classifications (e.g. 1=Pass, 0=Fail).
    """
    assert len(human_ratings) == len(llm_ratings)
    N = len(human_ratings)
    
    # Observed agreement
    p_o = np.mean(np.array(human_ratings) == np.array(llm_ratings))
    
    # Marginal probabilities
    p_h_pos = np.mean(human_ratings)
    p_h_neg = 1.0 - p_h_pos
    p_l_pos = np.mean(llm_ratings)
    p_l_neg = 1.0 - p_l_pos
    
    # Chance agreement
    p_e = (p_h_pos * p_l_pos) + (p_h_neg * p_l_neg)
    
    kappa = (p_o - p_e) / (1.0 - p_e + 1e-9)
    return p_o, p_e, kappa

# Verification
human_labels = [1, 1, 0, 1, 0, 0, 1, 1, 0, 1]
llm_labels   = [1, 1, 0, 1, 0, 1, 1, 1, 0, 1] # 9/10 agreement

po, pe, k = cohen_kappa(human_labels, llm_labels)
print(f"Observed Agreement (P_o): {po*100:.1f}%")
print(f"Expected Chance (P_e)   : {pe*100:.1f}%")
print(f"Cohen's Kappa (kappa)   : {k:.3f} (Values > 0.8 indicate near-perfect human alignment)")

## 3. High-Yield Interview Questions & Best Practices

### Q1: What is G-Eval and why does form-filling grading outperform single-number scoring?
**Answer:** G-Eval (Liu et al., 2023) breaks down qualitative evaluation into explicit algorithmic steps:
1. Define a detailed rubric with discrete grading criteria (e.g. 1=Severe Hallucination, 5=Completely Fact-Grounded).
2. Prompt the LLM judge to output **step-by-step reasoning (Chain-of-Thought)** before assigning a numeric grade.
3. Calculate the expected score using the output token probabilities of the grade tokens:
$$\text{Score} = \sum_{i=1}^5 i \times P(\text{token } i)$$
This produces continuous, calibrated scores that correlate significantly better with human experts than raw one-shot scores.

### Q2: How can verbosity bias be mitigated when comparing responses?
**Answer:**
1. **Length Normalization**: Penalize candidate responses that exceed the target brevity limit without contributing new unique factual propositions.
2. **Explicit System Prompt Rules**: Instruct the judge: *"Do not prefer longer responses. Penalize answers with redundant introductory or concluding pleasantries."*
3. **Fact-Extraction Intermediate Representation**: First extract atomic facts from both models into bullet points, then judge the factual bullets rather than the raw styled text.